# Supplemental: Polars — a modern DataFrame library

**CS1090A / AC209A — Module 1** *(optional enrichment)*

Lecture 2 mentioned that Pandas is no longer the only serious DataFrame library
in Python. This notebook gives you a working taste of **[Polars](https://pola.rs)** —
what's actually different, and when you might reach for it.

The point is *not* to replace Pandas (HW1 expects Pandas). It's to see that
"DataFrame" is a **concept** with more than one implementation, so your mental
model transfers.

Polars in three bullets:
- Written in **Rust**, built on **Apache Arrow** memory — fast and multi-threaded
  by default.
- An **expression API**: you describe *what* to compute (`pl.col('x').mean()`),
  not step-by-step *how*.
- **Lazy execution**: build a whole query, let an optimizer plan it, then run it
  once — the same idea that powers SQL databases and Spark.

In [1]:
import numpy as np
import pandas as pd
import polars as pl

print(f"pandas {pd.__version__}, polars {pl.__version__}")

pandas 2.3.3, polars 1.40.1


## 0. A toy dataset

To stay in HW1's world, we'll synthesize a table of *movie screenings*:
100,000 rows of (title, theater, genre, year, runtime, ticket price).

In [2]:
rng = np.random.default_rng(1090)
n = 100_000

theaters = ["The Brattle", "Coolidge Corner", "Somerville Theatre", "Harvard Film Archive"]
genres = ["Drama", "Comedy", "Horror", "Sci-Fi", "Documentary"]

screenings_pd = pd.DataFrame({
    "title": [f"Movie {i}" for i in rng.integers(0, 2_000, n)],
    "theater": rng.choice(theaters, n),
    "genre": rng.choice(genres, n),
    "year": rng.integers(1930, 2026, n),
    "runtime_min": rng.normal(110, 25, n).round().clip(45, 360),
    "price": rng.choice([9.0, 11.5, 13.0, 15.0], n),
})
screenings_pl = pl.from_pandas(screenings_pd)   # zero-ish copy via Arrow
screenings_pl.head(3)

title,theater,genre,year,runtime_min,price
str,str,str,i64,f64,f64
"""Movie 576""","""The Brattle""","""Documentary""",1942,101.0,9.0
"""Movie 1353""","""Somerville Theatre""","""Documentary""",2025,96.0,15.0
"""Movie 960""","""The Brattle""","""Documentary""",1988,90.0,13.0


## 1. The same question, side by side

*"Average runtime and total revenue per theater, for post-1980 films, sorted by
revenue."*

Read both versions and notice the difference in *style*: Pandas chains
**methods on objects**; Polars composes **expressions inside verbs**
(`filter`, `group_by`, `agg`).

In [3]:
# Pandas
(screenings_pd[screenings_pd.year > 1980]
    .groupby("theater")
    .agg(avg_runtime=("runtime_min", "mean"), revenue=("price", "sum"))
    .sort_values("revenue", ascending=False))

,avg_runtime,revenue
theater,,
The Brattle,110.019667,144279.0
Coolidge Corner,110.373063,142868.0
Harvard Film Archive,110.190611,141013.0
Somerville Theatre,109.763215,139568.5


In [4]:
# Polars
(screenings_pl
    .filter(pl.col("year") > 1980)
    .group_by("theater")
    .agg(
        avg_runtime=pl.col("runtime_min").mean(),
        revenue=pl.col("price").sum(),
    )
    .sort("revenue", descending=True))

theater,avg_runtime,revenue
str,f64,f64
"""The Brattle""",110.019667,144279.0
"""Coolidge Corner""",110.373063,142868.0
"""Harvard Film Archive""",110.190611,141013.0
"""Somerville Theatre""",109.763215,139568.5


Same answer, similar length. Two things Polars users tend to love:

- **No index.** A Polars frame is just columns — nothing like Pandas'
  index/reset_index dance.
- **Expressions compose.** `pl.col('price').sum()` is a value you can build,
  store in a variable, and reuse across queries.

## 2. Lazy mode: describe first, run once

Wrap a frame with `.lazy()` and nothing computes until `.collect()`. Polars
builds a **query plan** and optimizes it — pushing filters down before joins,
reading only needed columns, fusing operations. With `scan_csv`/`scan_parquet`
it can even skip reading data your query never touches.

In [5]:
lazy_query = (
    screenings_pl.lazy()
    .filter(pl.col("genre") == "Horror")
    .group_by("theater")
    .agg(screenings=pl.len(), avg_price=pl.col("price").mean())
    .sort("screenings", descending=True)
)

print(lazy_query.explain())   # the optimized plan — note the pushed-down filter

SORT BY [descending: [true]] [col("screenings")]
  AGGREGATE[maintain_order: false]
    [len().alias("screenings"), col("price").mean().alias("avg_price")] BY [col("theater")]
    FROM
    simple π 2/2 ["price", "theater"]
      FILTER [(col("genre")) == ("Horror")]
      FROM
        DF ["title", "theater", "genre", "year", ...]; PROJECT["price", "theater", "genre"] 3/6 COLUMNS


In [6]:
lazy_query.collect()   # ...and only now does any work happen

theater,screenings,avg_price
str,u32,f64
"""Coolidge Corner""",5105,12.155338
"""Somerville Theatre""",5025,12.126866
"""Harvard Film Archive""",4986,12.114822
"""The Brattle""",4972,12.175382


## 3. Is it actually faster?

On 100k rows both are instant; the gap shows up with more data and more cores.
A quick (unscientific) timing on our frame:

In [7]:
import time

def timeit(fn, reps=20):
    t0 = time.perf_counter()
    for _ in range(reps):
        fn()
    return (time.perf_counter() - t0) / reps * 1000

pd_ms = timeit(lambda: screenings_pd[screenings_pd.year > 1980]
               .groupby("theater")["price"].sum())
pl_ms = timeit(lambda: screenings_pl.filter(pl.col("year") > 1980)
               .group_by("theater").agg(pl.col("price").sum()))

print(f"pandas: {pd_ms:.2f} ms   polars: {pl_ms:.2f} ms   (per run, mean of 20)")

pandas: 16.70 ms   polars: 4.61 ms   (per run, mean of 20)


Your numbers will vary by machine — the honest summary is that Polars' advantage
grows with data size, core count, and query complexity, and can reach 5–50× on
large workloads. On small data, *both are fast enough and the difference is noise*.

## 4. Interop and file formats

Because both sit on (or near) Arrow memory, converting is cheap — use whichever
tool fits the task and hand off freely. Both read/write **Parquet**, the
columnar format from Lecture 2 that preserves dtypes (unlike CSV).

In [8]:
import os
os.makedirs("data", exist_ok=True)

# Polars -> Parquet -> Pandas round trip
screenings_pl.write_parquet("data/screenings.parquet")
back = pd.read_parquet("data/screenings.parquet")

print(type(back), back.shape)
print(back.dtypes.head(3))    # dtypes survived the round trip

<class 'pandas.core.frame.DataFrame'> (100000, 6)
title      object
theater    object
genre      object
dtype: object


## 5. When to reach for which

| Situation | Reasonable default |
|---|---|
| Course homework, most EDA, tons of Stack Overflow/LLM support | **Pandas** |
| Data bigger than a few GB, or a slow groupby-heavy pipeline | **Polars** |
| You want query optimization / out-of-core reads | **Polars (lazy + scan_*)** |
| A library you depend on expects one or the other | that one |

The deeper lesson: the *concepts* — tidy columns, filter → group → aggregate,
columnar formats — transfer across every DataFrame tool (Pandas, Polars, DuckDB,
Spark, SQL). Learn the concepts once; syntax is cheap (especially with an LLM at
your elbow).

**Exercises:**
1. Rewrite one of your HW1 Q4/Q5 Pandas manipulations in Polars. Which parts map
   one-to-one, and where did you need a genuinely different idiom?
2. Use `lazy_query.explain()` on a query with a filter *after* a group_by. Does
   the optimizer move it?
3. Compare `data/screenings.parquet` to the same frame written as CSV: file size,
   write time, and what happens to dtypes on re-read.